In [3]:
import os
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms, models
from PIL import Image
from tqdm import tqdm

In [4]:
# === CONFIG ===
class Config:
    IMAGE_SIZE = 224  # input size for ViT/AST
    BATCH_SIZE = 8
    EPOCHS = 50
    LEARNING_RATE = 1e-4
    DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    DATA_DIR = "generated_samples_valve/final_samples"

config = Config()

# === DATASET ===
class SpectrogramDataset(Dataset):
    def __init__(self, root_dir, transform=None):
        self.samples = []
        self.transform = transform

        for label_str in ["normal", "abnormal"]:
            label = 0 if label_str == "normal" else 1
            class_dir = os.path.join(root_dir, label_str)
            for file in os.listdir(class_dir):
                if file.endswith(".png"):
                    self.samples.append((os.path.join(class_dir, file), label))

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        img_path, label = self.samples[idx]
        image = Image.open(img_path).convert("RGB")
        if self.transform:
            image = self.transform(image)
        return image, label

# === TRANSFORMS ===
transform = transforms.Compose([
    transforms.Resize((config.IMAGE_SIZE, config.IMAGE_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.5]*3, std=[0.5]*3)
])

# === DATALOADER ===
dataset = SpectrogramDataset(config.DATA_DIR, transform=transform)
dataloader = DataLoader(dataset, batch_size=config.BATCH_SIZE, shuffle=True)

# === MODEL ===
class ASTModel(nn.Module):
    def __init__(self):
        super().__init__()
        self.backbone = models.vit_b_16(weights="IMAGENET1K_V1")

        # Get input features from the last Linear layer in the `heads` Sequential
        in_features = self.backbone.heads[-1].in_features
        
        # Replace the last Linear layer with a new one for binary classification
        self.backbone.heads[-1] = nn.Linear(in_features, 2)

    def forward(self, x):
        return self.backbone(x)

model = ASTModel().to(config.DEVICE)

# === TRAINING ===
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=config.LEARNING_RATE)

def train():
    model.train()
    for epoch in range(config.EPOCHS):
        total_loss = 0
        correct = 0
        total = 0
        for images, labels in tqdm(dataloader, desc=f"Epoch {epoch+1}/{config.EPOCHS}"):
            images, labels = images.to(config.DEVICE), labels.to(config.DEVICE)

            optimizer.zero_grad()
            outputs = model(images)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()

            total_loss += loss.item()
            preds = torch.argmax(outputs, dim=1)
            correct += (preds == labels).sum().item()
            total += labels.size(0)

        acc = correct / total * 100
        print(f"Epoch {epoch+1}: Loss={total_loss:.4f}, Accuracy={acc:.2f}%")

# === MAIN ===
if __name__ == '__main__':
    train()
    torch.save(model.state_dict(), "ast_model_synthetic.pth")

Epoch 1/50: 100%|██████████| 50/50 [00:16<00:00,  3.11it/s]


Epoch 1: Loss=38.7596, Accuracy=49.00%


Epoch 2/50: 100%|██████████| 50/50 [00:15<00:00,  3.18it/s]


Epoch 2: Loss=35.6908, Accuracy=49.00%


Epoch 3/50: 100%|██████████| 50/50 [00:17<00:00,  2.84it/s]


...

Epoch 48: Loss=34.8913, Accuracy=52.00%


Epoch 49/50: 100%|██████████| 50/50 [00:32<00:00,  1.54it/s]


Epoch 49: Loss=35.0824, Accuracy=49.00%


Epoch 50/50: 100%|██████████| 50/50 [00:31<00:00,  1.60it/s]


Epoch 50: Loss=34.8522, Accuracy=49.00%
